# Track C — 04. Fusions Scoring

> **TEMP SCORING — p0 values are placeholders pending CRISPR/GDSC calibration.
> k values are locked by feature shape. In-frame boost b=0.15 is locked.
> Logged in DECISIONS.md as pending.**

Converts `cleaned_track_data/fusions_gene_level.parquet` into a per-`(ensg_id, model_id)`
probability score `p_fusion` using a noisy-OR Hill-function model.

**Pipeline:**
1. Ordinal encode `max_confidence` (low=1, medium=2, high=3)
2. Per-gene percentile ranking of `best_ffpm` and `fusion_count`
   (`conf_ord` is NOT percentile-ranked — only 3 discrete values; percentile
   collapses to the same 3 outputs)
3. Hill function on each feature — k set by feature shape
4. Noisy-OR base score
5. Additive in-frame boost (b=0.15, locked — same as `driver_flag` in mutations)
6. Max-aggregate to one row per `(ensg_id, model_id)`

**Note:** fusions does NOT split into quality/burden columns.
`signature_discount` applies `m_fus` to the whole `p_fusion` score (`s_fus' = m_fus * s_fus`).
Single output column `p_fusion` is correct.

**Output:** `cleaned_track_data/fusions_scores.parquet`  
**Columns:** `ensg_id`, `model_id`, `p_fusion`, `conf_scored`, `ffpm_scored`, `recur_scored`

In [ ]:
import os
import sys

_notebook_dir = os.path.dirname(os.path.abspath('__file__'))
# src/Track - C/ -> up one level to src/, then into scripts/
sys.path.insert(0, os.path.join(_notebook_dir, '..', 'scripts'))

import numpy as np
import pandas as pd
from data_utils import REF

PROJECT_ROOT = os.path.abspath(os.path.join(REF, '..'))
DATA_DIR     = os.path.join(PROJECT_ROOT, 'cleaned_track_data')

IN_FUS   = os.path.join(DATA_DIR, 'fusions_gene_level.parquet')
IN_GENE  = os.path.join(REF, 'gene_lookup.parquet')
IN_CELL  = os.path.join(REF, 'cell_line_lookup.parquet')
OUT_PATH = os.path.join(DATA_DIR, 'fusions_scores.parquet')

# --- Hill-function hyperparameters (TEMP — pending CRISPR/GDSC calibration) ---
# feature      (p0,   k)    rationale
PARAMS_FUS = {
    'conf':  (2.0,  2.0),  # midpoint at 'medium'; discrete -> moderate k
    'ffpm':  (0.50, 1.5),  # percentile space; extreme right skew -> gentle k
    'recur': (0.50, 1.5),  # percentile space; 87% mass at 1 -> gentle k
}
INFRAME_BOOST = 0.15  # locked, same as driver_flag in mutations

CONF_MAP = {'low': 1, 'medium': 2, 'high': 3}

print('IN_FUS:  ', IN_FUS)
print('OUT_PATH:', OUT_PATH)

In [ ]:
df = pd.read_parquet(IN_FUS)

print(f'fusions_gene_level: {len(df):,} rows x {df.shape[1]} cols')
print()
print('dtypes:')
print(df.dtypes)
print()
print('max_confidence value counts:')
print(df['max_confidence'].value_counts())
print()
print('best_ffpm describe:')
print(df['best_ffpm'].describe(percentiles=[.25, .5, .75, .9, .95, .99]).round(4))
print()
print('fusion_count describe:')
print(df['fusion_count'].describe().round(4))
print()
n_inframe = int(df['any_in_frame'].sum())
print(f'any_in_frame True: {n_inframe:,} / {len(df):,}  ({100 * n_inframe / len(df):.1f}%)')

## 1. Feature encoding + per-gene percentile ranking

```
conf_ord   = CONF_MAP[max_confidence]          <- raw ordinal (1/2/3)
ffpm_pctl  = rank(pct=True) within ensg_id     <- normalises extreme right skew
recur_pctl = rank(pct=True) within ensg_id     <- separates recurrent from single-hit

Hill(x, p0, k) = x^k / (x^k + p0^k)

p_conf  = Hill(conf_ord,   p0=2.0, k=2.0)
p_ffpm  = Hill(ffpm_pctl,  p0=0.5, k=1.5)
p_recur = Hill(recur_pctl, p0=0.5, k=1.5)
```

Expected Hill outputs for `conf_ord` at these params:

| Confidence | conf_ord | p_conf |
|---|---|---|
| low | 1 | 0.20 |
| medium | 2 | 0.50 |
| high | 3 | 0.69 |

NaN inputs → `fillna(0)` before Hill so missing evidence contributes nothing to the OR.  
Audit flags (`conf_scored`, `ffpm_scored`, `recur_scored`) captured **before** fillna.

In [ ]:
def hill(x, p0, k):
    """Sigmoid-shaped Hill function mapping x -> (0, 1)."""
    xk = np.power(np.clip(x, 0, None), k)
    return xk / (xk + p0 ** k)


df = df.copy()

# a. Ordinal encode confidence — NaN if value not in CONF_MAP
df['conf_ord'] = df['max_confidence'].map(CONF_MAP).astype(float)

# b. Per-gene percentile ranking (conf_ord excluded — only 3 discrete values)
df['ffpm_pctl']  = df.groupby('ensg_id')['best_ffpm'].rank(pct=True)
df['recur_pctl'] = df.groupby('ensg_id')['fusion_count'].rank(pct=True)

# c. Audit flags — captured before fillna so NaN inputs are traceable
df['conf_scored']  = df['max_confidence'].notna()
df['ffpm_scored']  = df['best_ffpm'].notna()
df['recur_scored'] = df['fusion_count'].notna()

# d. Hill on each feature (NaN -> 0 before Hill; missing = no contribution to OR)
df['p_conf']  = hill(df['conf_ord'].fillna(0),   *PARAMS_FUS['conf'])
df['p_ffpm']  = hill(df['ffpm_pctl'].fillna(0),  *PARAMS_FUS['ffpm'])
df['p_recur'] = hill(df['recur_pctl'].fillna(0), *PARAMS_FUS['recur'])

print('p_conf describe:')
print(df['p_conf'].describe().round(4))
print()
print('p_ffpm describe:')
print(df['p_ffpm'].describe().round(4))
print()
print('p_recur describe:')
print(df['p_recur'].describe().round(4))

## 2. Noisy-OR base score + in-frame boost

```
p_base       = 1 - (1 - p_conf) * (1 - p_ffpm) * (1 - p_recur)
inframe_flag  = any_in_frame.astype(int)
p_fusion      = min(p_base + 0.15 * inframe_flag, 1.0)
```

Then max-aggregate over `(ensg_id, model_id)` pairs.  
Audit flags aggregate via `any` — True if any row for that pair had a scored value.

In [ ]:
p_base = 1.0 - (1.0 - df['p_conf']) * (1.0 - df['p_ffpm']) * (1.0 - df['p_recur'])

inframe_flag = df['any_in_frame'].fillna(False).astype(int)

df['p_fusion'] = (p_base + INFRAME_BOOST * inframe_flag).clip(upper=1.0)

# Max-aggregate to one row per (ensg_id, model_id)
scores = (
    df.groupby(['ensg_id', 'model_id'], sort=False)
    .agg(
        p_fusion     = ('p_fusion',     'max'),
        conf_scored  = ('conf_scored',  'any'),
        ffpm_scored  = ('ffpm_scored',  'any'),
        recur_scored = ('recur_scored', 'any'),
    )
    .reset_index()
)

print(f'input rows:  {len(df):,}')
print(f'output rows: {len(scores):,}  (one per ensg_id / model_id pair)')
print(f'distinct genes:      {scores["ensg_id"].nunique():,}')
print(f'distinct cell lines: {scores["model_id"].nunique():,}')
print()
print(f'conf_scored  True: {scores["conf_scored"].sum():,}  ({100 * scores["conf_scored"].mean():.1f}%)')
print(f'ffpm_scored  True: {scores["ffpm_scored"].sum():,}  ({100 * scores["ffpm_scored"].mean():.1f}%)')
print(f'recur_scored True: {scores["recur_scored"].sum():,}  ({100 * scores["recur_scored"].mean():.1f}%)')
print()
print('p_fusion describe:')
print(scores['p_fusion'].describe().round(4))
print()
print(f'rows with in-frame boost: {(inframe_flag == 1).sum():,}')
print(f'rows capped at 1.0:       {(df["p_fusion"] == 1.0).sum():,}')

## 3. Write output

In [ ]:
scores[['ensg_id', 'model_id', 'p_fusion', 'conf_scored', 'ffpm_scored', 'recur_scored']].to_parquet(OUT_PATH, index=False)
print(f'written: {OUT_PATH}')
print(f'rows:    {len(scores):,}')
print(f'columns: {list(scores.columns)}')

## 4. Validation spot-check

Resolve canonical model IDs from `cell_line_lookup` then check known fusion-driven lines.

| Gene | Cell line | Expected |
|------|-----------|----------|
| ALK (ENSG00000171094) | NCI-H2228 | > 0.6 — EML4-ALK NSCLC |
| ABL1 (ENSG00000097007) | K562 | > 0.6 — BCR-ABL1 CML |
| BRAF (ENSG00000157764) | A375 | MISSING — BRAF V600E is a point mutation, not a fusion; absence is biologically correct |

In [ ]:
clk = pd.read_parquet(IN_CELL)

name_to_model = (
    clk[['model_id', 'cell_line_name']]
      .dropna(subset=['cell_line_name'])
      .drop_duplicates(subset=['cell_line_name'])
      .set_index('cell_line_name')['model_id']
)

ANCHORS = [
    ('ENSG00000171094', 'ALK',  'nci-h2228', 'NCI-H2228', 'EML4-ALK NSCLC   -- expect > 0.6'),
    ('ENSG00000097007', 'ABL1', 'k-562',     'K562',      'BCR-ABL1 CML     -- expect > 0.6'),
    ('ENSG00000157764', 'BRAF', 'a-375',     'A375',      'V600E point mut  -- expect ABSENT'),
]

scores_idx = scores.set_index(['ensg_id', 'model_id'])['p_fusion']

print(f'{"Gene":<6}  {"Cell line":<10}  {"model_id":<12}  {"p_fusion":>10}  status')
print('-' * 80)
for ensg, gene, lookup_name, display_name, note in ANCHORS:
    model_id = name_to_model.get(lookup_name, None)
    if model_id is None:
        print(f'{gene:<6}  {display_name:<10}  {"NOT IN LOOKUP":<12}  {"N/A":>10}  SKIP')
        continue
    score = scores_idx.get((ensg, model_id), float('nan'))
    if np.isnan(score):
        suffix = '  (expected -- V600E is not a fusion)' if gene == 'BRAF' else ''
        status = 'MISSING FROM DATA' + suffix
    elif score > 0.6:
        status = 'OK (> 0.6)'
    else:
        status = 'LOW -- check inputs'
    print(f'{gene:<6}  {display_name:<10}  {model_id:<12}  {score:>10.4f}  {status}')

print()
print('Note: BRAF/A375 absent is biologically correct -- BRAF V600E is a point mutation.')
print('      Its dependency signal will appear in mutations scoring, not fusions.')